In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 52.0 MB/s eta 0:00:00


In [2]:
import nltk
from gensim.models import Word2Vec
import numpy as np

In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
text = """
Formula one racing is a fast motorsport.
Racing drivers compete in different circuits.
Teams develop fast racing cars.
Drivers use tyres during every race.
"""

In [5]:
sentences = nltk.sent_tokenize(text.lower())
print(sentences)

['\nformula one racing is a fast motorsport.', 'racing drivers compete in different circuits.', 'teams develop fast racing cars.', 'drivers use tyres during every race.']


In [6]:
tokens = [
    nltk.word_tokenize(sentence)
    for sentence in sentences
]

print(tokens)

[['formula', 'one', 'racing', 'is', 'a', 'fast', 'motorsport', '.'], ['racing', 'drivers', 'compete', 'in', 'different', 'circuits', '.'], ['teams', 'develop', 'fast', 'racing', 'cars', '.'], ['drivers', 'use', 'tyres', 'during', 'every', 'race', '.']]


In [7]:
model = Word2Vec(
    tokens, vector_size=50,
    window=3, min_count=1
)

print("Word2Vec model trained")

Word2Vec model trained


In [8]:
vector = model.wv["racing"]

print("Vector for racing:")
print(vector)

Vector for racing:
[-0.01631443  0.0089921  -0.00827236  0.00164808  0.01699637 -0.00892621
  0.00903411 -0.01357697 -0.00709871  0.01879543 -0.00315303  0.00064091
 -0.00827942 -0.01536484 -0.00301734  0.00494042 -0.00177233  0.0110706
 -0.0054879   0.00452175  0.01091091  0.01669419 -0.00290624 -0.0184155
  0.00874186  0.00114644  0.01488372 -0.00162512 -0.00527517 -0.01750899
 -0.00171385  0.00565115  0.01080325  0.0141053  -0.01140848  0.00371541
  0.01217841 -0.00959683 -0.00621638  0.01359302  0.0032642   0.00037906
  0.00695015  0.00043325  0.01923972  0.01012099 -0.01783366 -0.01408323
  0.00180313  0.01278467]


In [9]:
similar = model.wv.most_similar(
    "racing", topn=5
)

print(similar)

[('motorsport', 0.22982969880104065), ('one', 0.16084668040275574), ('is', 0.1487700492143631), ('race', 0.12486038357019424), ('circuits', 0.08066050708293915)]


In [10]:
score = model.wv.similarity(
    "racing", "drivers"
)

print("Similarity:", score)

Similarity: 0.010987813


In [26]:
from collections import defaultdict

vocab = sorted(set(word for s in tokens for word in s))
word_id = {w:i for i,w in enumerate(vocab)}

cooccur = defaultdict(float)
window = 2

In [27]:
for sentence in tokens:
    for i, word in enumerate(sentence):
        for j in range(max(0,i-window), min(len(sentence),i+window+1)):
            if i != j:
                cooccur[(word_id[word], word_id[sentence[j]])] += 1

print("Vocabulary:", vocab)
print("Co-occurrence pairs:", len(cooccur))

Vocabulary: ['.', 'a', 'cars', 'circuits', 'compete', 'develop', 'different', 'drivers', 'during', 'every', 'fast', 'formula', 'in', 'is', 'motorsport', 'one', 'race', 'racing', 'teams', 'tyres', 'use']
Co-occurrence pairs: 88


In [28]:
W = np.random.rand(len(vocab), 20)
U = np.random.rand(len(vocab), 20)

for _ in range(100):
    for (i,j), x in cooccur.items():
        error = np.dot(W[i], U[j]) - np.log(x)
        W[i] -= 0.01 * error * U[j]
        U[j] -= 0.01 * error * W[i]

print("GloVe embeddings trained")

GloVe embeddings trained


In [29]:
glove_vectors = W + U

idx = word_id["racing"]
print("GloVe vector for 'racing':")
print(glove_vectors[idx])

GloVe vector for 'racing':
[ 0.12490881  0.28300703  0.21001586  0.32318653 -0.5537687   0.03246192
  0.25668475 -0.26553073 -0.16925934  0.52895585 -0.11569379  0.15587695
 -0.0725173   0.19336558  0.68773972  0.32715517 -0.3229828  -0.02108172
  0.82719275  0.61437241]


In [30]:
def similarity(word1, word2):
    a = glove_vectors[word_id[word1]]
    b = glove_vectors[word_id[word2]]
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

print("Similarity:", similarity("racing", "drivers"))

Similarity: 0.32630185962200986
